In [3]:
# REFORESTACIÓN SPECTRUM PROTOTIPO POR MARIA JOSE CASTRO
#Video Microsoft https://www.youtube.com/watch?v=stNlw_CFklc

import piplite
await piplite.install(['ipyleaflet', 'ipywidgets', 'shapely', 'requests'])

import copy
import math
import re
import unicodedata
from datetime import date, datetime

import requests
import ipywidgets as widgets

from IPython.display import display, clear_output
from ipyleaflet import (
    AwesomeIcon,
    CircleMarker,
    DrawControl,
    FullScreenControl,
    GeoJSON,
    LayerGroup,
    LayersControl,
    Map,
    Marker,
    Polygon as LeafletPolygon,
    Polyline,
    ScaleControl,
    basemaps,
)
from shapely.geometry import shape, Polygon, GeometryCollection, mapping
from shapely.ops import unary_union


# 1. DATOS DE IMPACTO POR TIPO DE ÁRBOL
# Los valores representan una estimación de metros cuadrados por tipo de arbol
tree_impact_data = {
    "Pino": 9,
    "Ciprés": 8,
    "Encino": 15,
    "Caoba": 18,
    "Cedro": 16,
    "Matilisguate": 12,
    "Conacaste": 20,
    "Eucalipto": 9,
    "Hormigo": 9,
    "Nance": 12,
}


# 1.1 SUPERFICIE CONSTRUIDA POR SPECTRUM
# 1 manzana = 7050.2 metros cuadrados
    #vivenda 254 manzanas
    #propiedades: 20 manzanas miraflores, portales 6, naranjo 6, oakland 4.5, margaritas 1.5 y 
    #synergy 704 

M2_POR_MANZANA = 7050.2

SUPERFICIE_CONSTRUIDA_MANZANAS = {
    "Vivienda": 254,
    "Propiedades": 38,
    "Synergy": 704,
}

DETALLE_PROPIEDADES_MANZANAS = {
    "Miraflores": 20,
    "Portales": 6,
    "Naranjo": 6,
    "Oakland": 4.5,
    "Centro Gerencial Margaritas": 1.5,
}

COLORES_COMPARACION = {
    "Vivienda": "#173B57",
    "Propiedades": "#2E8B57",
    "Synergy": "#0B5D3B",
}


def calcular_superficie_construida():
    """Convierte las superficies construidas de manzanas a m²."""
    superficies = {}

    for categoria, manzanas in SUPERFICIE_CONSTRUIDA_MANZANAS.items():
        superficies[categoria] = {
            "manzanas": manzanas,
            "m2": manzanas * M2_POR_MANZANA,
        }

    return superficies


def calcular_porcentaje_equivalencia(area_reforestada_m2, area_construida_m2):
    """Compara dos superficies y devuelve su porcentaje de equivalencia."""
    if area_construida_m2 <= 0:
        return 0

    return (area_reforestada_m2 / area_construida_m2) * 100


def crear_barra_progreso(porcentaje, color):
    """Crea una barra HTML. El ancho visual se limita a 100%."""
    ancho_visual = max(0, min(porcentaje, 100))

    return f"""
    <div style="
        width:100%;
        height:15px;
        background:#E6EBEE;
        border-radius:999px;
        overflow:hidden;
        margin:8px 0 5px 0;
    ">
        <div style="
            width:{ancho_visual:.2f}%;
            height:100%;
            background:{color};
            border-radius:999px;
            transition:width 0.5s ease;
        "></div>
    </div>
    """


def crear_panel_comparacion_superficies(total_reforestado_m2):
    """Genera el panel visual de comparación contra lo construido."""
    superficies = calcular_superficie_construida()
    total_construido_m2 = sum(
        datos["m2"] for datos in superficies.values()
    )
    total_construido_manzanas = sum(
        datos["manzanas"] for datos in superficies.values()
    )

    porcentaje_total = calcular_porcentaje_equivalencia(
        total_reforestado_m2,
        total_construido_m2,
    )
    equivalencia_manzanas = total_reforestado_m2 / M2_POR_MANZANA
    diferencia_m2 = total_reforestado_m2 - total_construido_m2

    if diferencia_m2 >= 0:
        mensaje_diferencia = (
            f"La superficie reforestada supera la superficie construida "
            f"por <b>{diferencia_m2:,.0f} m²</b>."
        )
    else:
        mensaje_diferencia = (
            f"Faltan <b>{abs(diferencia_m2):,.0f} m²</b> para alcanzar "
            f"una equivalencia de superficie del 100%."
        )

    tarjetas_categorias = ""

    for categoria, datos in superficies.items():
        porcentaje = calcular_porcentaje_equivalencia(
            total_reforestado_m2,
            datos["m2"],
        )
        color = COLORES_COMPARACION[categoria]

        tarjetas_categorias += f"""
        <div style="
            flex:1 1 230px;
            background:white;
            border:1px solid #D9E1E5;
            border-top:5px solid {color};
            border-radius:10px;
            padding:14px;
            box-shadow:0 2px 8px rgba(23,59,87,0.08);
        ">
            <div style="
                color:{color};
                font-size:15px;
                font-weight:bold;
                margin-bottom:5px;
            ">
                {categoria}
            </div>

            <div style="font-size:25px; font-weight:bold; color:#263238;">
                {porcentaje:,.2f}%
            </div>

            <div style="font-size:12px; color:#607D8B;">
                equivalencia de la superficie reforestada
            </div>

            {crear_barra_progreso(porcentaje, color)}

            <div style="font-size:12px; color:#455A64; line-height:1.45;">
                Construido: <b>{datos["manzanas"]:,.1f} mz</b><br>
                Equivalente a: <b>{datos["m2"]:,.0f} m²</b>
            </div>
        </div>
        """

    detalle_propiedades = " · ".join(
        f"{nombre}: {manzanas:g} mz"
        for nombre, manzanas in DETALLE_PROPIEDADES_MANZANAS.items()
    )

    return f"""
    <div style="
        max-width:950px;
        margin:0 0 18px 0;
        padding:18px;
        background:linear-gradient(135deg, #F4FBF6, #EEF4F7);
        border:1px solid #C9DDD1;
        border-radius:12px;
        font-family:Arial;
        box-shadow:0 3px 12px rgba(11,93,59,0.10);
    ">
        <div style="
            display:flex;
            flex-wrap:wrap;
            justify-content:space-between;
            align-items:flex-start;
            gap:16px;
        ">
            <div>
                <div style="
                    color:#0B5D3B;
                    font-weight:bold;
                    font-size:16px;
                ">
                    Comparación territorial acumulada
                </div>

                <div style="
                    font-size:34px;
                    font-weight:bold;
                    color:#173B57;
                    margin-top:5px;
                ">
                    {total_reforestado_m2:,.0f} m²
                </div>

                <div style="font-size:13px; color:#546E7A;">
                    superficie real dibujada y registrada
                </div>
            </div>

            <div style="
                background:#173B57;
                color:white;
                padding:12px 16px;
                border-radius:10px;
                min-width:210px;
                text-align:center;
            ">
                <div style="font-size:12px; opacity:0.85;">
                    Equivalencia en manzanas
                </div>
                <div style="font-size:28px; font-weight:bold;">
                    {equivalencia_manzanas:,.2f} mz
                </div>
            </div>
        </div>

        <div style="margin-top:18px;">
            <div style="
                display:flex;
                justify-content:space-between;
                flex-wrap:wrap;
                gap:5px;
                font-size:13px;
                color:#37474F;
            ">
                <span>
                    <b>Área construida total:</b>
                    {total_construido_manzanas:,.1f} mz
                    ({total_construido_m2:,.0f} m²)
                </span>
                <span style="color:#0B5D3B; font-weight:bold;">
                    {porcentaje_total:,.2f}% alcanzado
                </span>
            </div>

            {crear_barra_progreso(porcentaje_total, "#0B5D3B")}

            <div style="font-size:13px; color:#455A64;">
                {mensaje_diferencia}
            </div>
        </div>

        <div style="
            display:flex;
            flex-wrap:wrap;
            gap:12px;
            margin-top:18px;
        ">
            {tarjetas_categorias}
        </div>

        <div style="
            background:white;
            border-left:4px solid #2E8B57;
            padding:10px 12px;
            margin-top:14px;
            border-radius:5px;
            font-size:12px;
            color:#455A64;
            line-height:1.5;
        ">
            <b>Detalle de propiedades:</b> {detalle_propiedades}.<br>
            <b>Interpretación:</b> esta métrica compara únicamente superficies.
            No representa por sí sola una compensación de carbono o de impactos
            ambientales de la construcción.
        </div>
    </div>
    """


# 1.2 FASES DE SEGUIMIENTO DE LA REFORESTACIÓN
FASES_REFORESTACION = {
    1: {
        "nombre": "1. Establecimiento",
        "temporalidad": "Meses 1 a 12",
        "objetivo": "Evaluar adaptación inicial.",
        "indicadores": [
            "Porcentaje de supervivencia",
            "Causas de mortalidad",
        ],
        "color_hex": "#8BC34A",
        "marker_color": "lightgreen",
    },
    2: {
        "nombre": "2. Desarrollo",
        "temporalidad": "Años 2 a 3",
        "objetivo": "Evaluar crecimiento y salud.",
        "indicadores": [
            "Altura y diámetro del tallo",
            "Estado fitosanitario (plagas)",
            "Control de malezas",
        ],
        "color_hex": "#2E8B57",
        "marker_color": "green",
    },
    3: {
        "nombre": "3. Consolidación",
        "temporalidad": "Años 4 a 5+",
        "objetivo": "Verificar autosuficiencia.",
        "indicadores": [
            "Incremento Medio Anual",
            "Retorno de biodiversidad",
            "Regeneración natural",
        ],
        "color_hex": "#0B5D3B",
        "marker_color": "darkgreen",
    },
}


def convertir_a_fecha(valor):
    """Convierte una fecha guardada como texto o datetime a date."""
    if isinstance(valor, datetime):
        return valor.date()

    if isinstance(valor, date):
        return valor

    if isinstance(valor, str):
        return datetime.strptime(valor, "%Y-%m-%d").date()

    raise ValueError("La fecha del proyecto no tiene un formato válido.")


def meses_completos_transcurridos(fecha_inicio, fecha_referencia=None):
    """Calcula meses completos entre dos fechas sin dependencias externas."""
    fecha_inicio = convertir_a_fecha(fecha_inicio)
    fecha_referencia = fecha_referencia or date.today()

    meses = (
        (fecha_referencia.year - fecha_inicio.year) * 12
        + fecha_referencia.month
        - fecha_inicio.month
    )

    if fecha_referencia.day < fecha_inicio.day:
        meses -= 1

    return max(0, meses)


def calcular_fase_reforestacion(fecha_proyecto, fecha_referencia=None):
    """Devuelve la fase actual y la antigüedad del proyecto."""
    fecha_proyecto = convertir_a_fecha(fecha_proyecto)
    fecha_referencia = fecha_referencia or date.today()

    if fecha_proyecto > fecha_referencia:
        raise ValueError("La fecha del proyecto no puede estar en el futuro.")

    meses = meses_completos_transcurridos(
        fecha_proyecto,
        fecha_referencia,
    )
    dias = (fecha_referencia - fecha_proyecto).days

    # Año 1: meses 1 a 12. Años 2 y 3: desde el primer aniversario hasta antes del tercero. Año 4+: consolidación.
    if meses < 12:
        numero_fase = 1
    elif meses < 36:
        numero_fase = 2
    else:
        numero_fase = 3

    datos_fase = FASES_REFORESTACION[numero_fase].copy()
    datos_fase["numero"] = numero_fase
    datos_fase["meses_transcurridos"] = meses
    datos_fase["dias_transcurridos"] = dias

    if dias == 0:
        antiguedad = "Realizado hoy"
    elif meses == 0:
        antiguedad = f"{dias} día" if dias == 1 else f"{dias} días"
    else:
        anios = meses // 12
        meses_restantes = meses % 12
        partes = []

        if anios:
            partes.append(
                f"{anios} año" if anios == 1 else f"{anios} años"
            )

        if meses_restantes:
            partes.append(
                f"{meses_restantes} mes"
                if meses_restantes == 1
                else f"{meses_restantes} meses"
            )

        antiguedad = " y ".join(partes)

    datos_fase["antiguedad"] = antiguedad
    return datos_fase


def formatear_fecha(fecha_proyecto):
    """Presenta una fecha como día/mes/año."""
    return convertir_a_fecha(fecha_proyecto).strftime("%d/%m/%Y")



# 2. DESCARGAR ZONAS DE LA CIUDAD
ARCGIS_WEBMAP_ID = "73bcb606bd1f415081f04c8c75a377b4"
ARCGIS_ITEM_DATA_URL = (
    "https://www.arcgis.com/sharing/rest/content/items/"
    f"{ARCGIS_WEBMAP_ID}/data"
)

ZONAS_VALIDAS = [
    1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,
    14, 15, 16, 17, 18, 19, 21, 24, 25,
]


def normalizar_texto(texto):
    """Convierte un texto a mayúsculas y elimina tildes."""
    texto = str(texto).strip()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return texto.upper()


def recorrer_capas(capas):
    """Recorre capas y subcapas de un mapa web de ArcGIS."""
    for capa in capas or []:
        yield capa

        subcapas = capa.get("layers", [])
        if subcapas:
            yield from recorrer_capas(subcapas)


def obtener_wkid(capa_interna, feature_set, geometria):
    """Busca el sistema de coordenadas usado por una Feature Collection."""
    candidatos = [
        geometria.get("spatialReference", {}),
        feature_set.get("spatialReference", {}),
        capa_interna.get("layerDefinition", {}).get("spatialReference", {}),
        capa_interna.get("layerDefinition", {}).get("extent", {}).get(
            "spatialReference", {}
        ),
    ]

    for referencia in candidatos:
        wkid = referencia.get("latestWkid", referencia.get("wkid"))
        if wkid is not None:
            return int(wkid)

    return 4326


def convertir_xy_a_wgs84(x, y, wkid):
    """Convierte coordenadas geográficas o Web Mercator a WGS84."""
    x = float(x)
    y = float(y)

    # Coordenadas geográficas WGS84 y sistemas equivalentes para este uso.
    if wkid in {4326, 4269}:
        return [x, y]

    # Web Mercator Auxiliary Sphere.
    if wkid in {3857, 102100, 102113, 900913}:
        longitud = (x / 20037508.34) * 180.0
        latitud = (y / 20037508.34) * 180.0
        latitud = (
            180.0
            / math.pi
            * (
                2.0
                * math.atan(math.exp(latitud * math.pi / 180.0))
                - math.pi / 2.0
            )
        )
        return [longitud, latitud]

    if -180 <= x <= 180 and -90 <= y <= 90:
        return [x, y]

    raise RuntimeError(
        f"El sistema de coordenadas WKID {wkid} no está soportado."
    )


def convertir_poligono_esri_a_shapely(geometria, wkid):
    """Convierte los anillos de un polígono Esri JSON a Shapely."""
    anillos = geometria.get("rings", [])
    poligonos_anillo = []

    for anillo in anillos:
        coordenadas = []

        for punto in anillo:
            if len(punto) < 2:
                continue

            coordenadas.append(
                convertir_xy_a_wgs84(punto[0], punto[1], wkid)
            )

        if len(coordenadas) < 3:
            continue

        if coordenadas[0] != coordenadas[-1]:
            coordenadas.append(coordenadas[0])

        if len(coordenadas) < 4:
            continue

        poligono = Polygon(coordenadas)

        if not poligono.is_valid:
            poligono = poligono.buffer(0)

        if not poligono.is_empty:
            poligonos_anillo.append(poligono)

    if not poligonos_anillo:
        return None

    # ArcGIS dibuja polígonos con la regla par-impar. 
    geometria_final = GeometryCollection()

    for poligono in poligonos_anillo:
        geometria_final = geometria_final.symmetric_difference(poligono)

    if not geometria_final.is_valid:
        geometria_final = geometria_final.buffer(0)

    return geometria_final if not geometria_final.is_empty else None


def obtener_geojson_desde_feature_collection(capa):
    """Convierte una Feature Collection incrustada de Esri JSON a GeoJSON."""
    feature_collection = capa.get("featureCollection")
    if not feature_collection:
        return None

    features_geojson = []

    for capa_interna in feature_collection.get("layers", []):
        feature_set = capa_interna.get("featureSet", {})
        features = feature_set.get("features", [])

        geometry_type = feature_set.get(
            "geometryType",
            capa_interna.get("layerDefinition", {}).get("geometryType", ""),
        )

        if not features or "Polygon" not in geometry_type:
            continue

        for feature in features:
            geometria_esri = feature.get("geometry", {})
            wkid = obtener_wkid(capa_interna, feature_set, geometria_esri)

            geometria_shapely = convertir_poligono_esri_a_shapely(
                geometria_esri,
                wkid,
            )

            if geometria_shapely is None:
                continue

            features_geojson.append(
                {
                    "type": "Feature",
                    "properties": feature.get("attributes", {}).copy(),
                    "geometry": mapping(geometria_shapely),
                }
            )

    if not features_geojson:
        return None

    return {
        "type": "FeatureCollection",
        "features": features_geojson,
    }


def descargar_geojson_arcgis(url_capa):
    """Consulta una capa de ArcGIS y devuelve sus elementos como GeoJSON."""
    query_url = f"{url_capa.rstrip('/')}/query"

    parametros = {
        "where": "1=1",
        "outFields": "*",
        "returnGeometry": "true",
        "outSR": "4326",
        "f": "geojson",
    }

    respuesta = requests.get(query_url, params=parametros, timeout=90)
    respuesta.raise_for_status()

    geojson = respuesta.json()

    if geojson.get("type") != "FeatureCollection":
        mensaje = geojson.get("error", geojson)
        raise RuntimeError(
            "ArcGIS no devolvió un FeatureCollection válido. "
            f"Respuesta: {mensaje}"
        )

    if not geojson.get("features"):
        raise RuntimeError("La capa de zonas no devolvió polígonos.")

    return geojson


def encontrar_capa_de_zonas(datos_mapa):
    """Localiza automáticamente la capa poligonal de zonas."""
    capas = list(recorrer_capas(datos_mapa.get("operationalLayers", [])))

    # Primera opción: buscar el título exacto o muy similar.
    for capa in capas:
        titulo = normalizar_texto(capa.get("title", ""))

        if "ZONAS DE LA CIUDAD DE GUATEMALA" in titulo:
            return capa

    # Segunda opción: cualquier capa cuyo título incluya zonas y Guatemala.
    for capa in capas:
        titulo = normalizar_texto(capa.get("title", ""))

        if "ZONA" in titulo and "GUATEMALA" in titulo:
            return capa

    titulos_disponibles = [
        capa.get("title", "Capa sin nombre")
        for capa in capas
    ]

    raise RuntimeError(
        "No se encontró la capa de zonas dentro del mapa web. "
        f"Capas disponibles: {titulos_disponibles}"
    )


def extraer_numero_zona(valor):
    """Extrae un número de zona válido desde textos o números."""
    if valor is None or isinstance(valor, bool):
        return None

    if isinstance(valor, (int, float)):
        numero = int(valor)
        return numero if numero in ZONAS_VALIDAS else None

    texto = normalizar_texto(valor)

    # Busca expresiones como ZONA 1, ZONA_01 o Z-1.
    patrones = [
        r"\bZONA\D{0,5}(\d{1,2})\b",
        r"\bZ\D{0,3}(\d{1,2})\b",
        r"^0*(\d{1,2})$",
    ]

    for patron in patrones:
        coincidencia = re.search(patron, texto)
        if coincidencia:
            numero = int(coincidencia.group(1))
            if numero in ZONAS_VALIDAS:
                return numero

    return None


def identificar_zona_feature(feature):
    """Identifica la zona usando los atributos de un elemento GeoJSON."""
    propiedades = feature.get("properties", {})

    # Primero revisa campos cuyo nombre sugiera que contienen la zona.
    claves_prioritarias = [
        clave
        for clave in propiedades
        if "ZONA" in normalizar_texto(clave)
    ]

    for clave in claves_prioritarias:
        numero = extraer_numero_zona(propiedades.get(clave))
        if numero is not None:
            return numero

    # Como respaldo, revisa todos los valores.
    for valor in propiedades.values():
        numero = extraer_numero_zona(valor)
        if numero is not None:
            return numero

    return None


def preparar_geojson_zonas(geojson_original):
    """Normaliza los atributos y crea un lookup de Zona -> polígono."""
    features_validas = []
    zone_lookup = {}

    for feature in geojson_original.get("features", []):
        if not feature.get("geometry"):
            continue

        numero_zona = identificar_zona_feature(feature)
        if numero_zona is None:
            continue

        nombre_zona = f"Zona {numero_zona}"

        feature.setdefault("properties", {})
        feature["properties"]["zona_numero"] = numero_zona
        feature["properties"]["zona_nombre"] = nombre_zona

        # En caso de que una zona venga fragmentada en varios polígonos, se guardan todos y luego se unifican.
        features_validas.append(feature)

    if not features_validas:
        ejemplo_claves = []
        if geojson_original.get("features"):
            ejemplo_claves = list(
                geojson_original["features"][0].get("properties", {}).keys()
            )

        raise RuntimeError(
            "Se descargaron polígonos, pero no se pudo identificar el "
            "campo de zona. Campos encontrados: "
            f"{ejemplo_claves}"
        )

    # Agrupar y unificar geometrías por zona.
    features_unificadas = []

    for numero_zona in ZONAS_VALIDAS:
        features_zona = [
            feature
            for feature in features_validas
            if feature["properties"]["zona_numero"] == numero_zona
        ]

        if not features_zona:
            continue

        geometrias = [shape(feature["geometry"]) for feature in features_zona]
        geometria_unificada = unary_union(geometrias)

        feature_unificada = {
            "type": "Feature",
            "properties": {
                "zona_numero": numero_zona,
                "zona_nombre": f"Zona {numero_zona}",
            },
            "geometry": geometria_unificada.__geo_interface__,
        }

        features_unificadas.append(feature_unificada)
        zone_lookup[f"Zona {numero_zona}"] = feature_unificada

    zonas_faltantes = [
        numero
        for numero in ZONAS_VALIDAS
        if f"Zona {numero}" not in zone_lookup
    ]

    if zonas_faltantes:
        print(
            "Advertencia: la fuente geográfica no incluyó estas zonas:",
            zonas_faltantes,
        )

    geojson_limpio = {
        "type": "FeatureCollection",
        "features": features_unificadas,
    }

    return geojson_limpio, zone_lookup


def cargar_zonas_ciudad_guatemala():
    """Descarga el mapa web y devuelve los polígonos normalizados."""
    print("Cargando mapa de zonas de la Ciudad de Guatemala...")

    respuesta_mapa = requests.get(
        ARCGIS_ITEM_DATA_URL,
        params={"f": "json"},
        timeout=60,
    )
    respuesta_mapa.raise_for_status()
    datos_mapa = respuesta_mapa.json()

    capa_zonas = encontrar_capa_de_zonas(datos_mapa)

    url_capa = capa_zonas.get("url")

    if url_capa:
        geojson_original = descargar_geojson_arcgis(url_capa)
    else:
        geojson_original = obtener_geojson_desde_feature_collection(capa_zonas)

    if geojson_original is None:
        raise RuntimeError(
            "La capa de zonas fue encontrada, pero no contiene una URL "
            "pública ni una colección geográfica utilizable."
        )

    geojson_limpio, lookup = preparar_geojson_zonas(geojson_original)

    print(
        "Mapa cargado correctamente:",
        len(lookup),
        "zonas disponibles.",
    )

    return geojson_limpio, lookup


try:
    geojson_zonas, zone_lookup = cargar_zonas_ciudad_guatemala()
except Exception as error:
    raise RuntimeError(
        "No fue posible cargar los límites de las zonas. Verifica tu "
        "conexión a internet y vuelve a ejecutar la celda. Detalle: "
        f"{error}"
    ) from error


zone_names = sorted(
    zone_lookup.keys(),
    key=lambda nombre: int(nombre.split()[-1]),
)



# 2.1 SECTORES DEL MASTER PLAN DE SYNERGY
# =========================================================
# Los polígonos se digitalizaron de forma aproximada a partir de la captura
# de Google Earth compartida. 
# Para una versión catastral exacta, reemplazar esta sección por un archivo
# KML/KMZ o GeoJSON exportado desde Google Earth, QGIS o ArcGIS.

TRANSFORMACION_SYNERGY = {
    "lon_x": 3.32620412e-05,
    "lon_y": -7.39465983e-06,
    "lon_c": -90.7990610,
    "lat_x": 1.78197706e-07,
    "lat_y": -3.11809558e-05,
    "lat_c": 14.3510306,
}

# Vértices aproximados tomados sobre la imagen de referencia de 1242 x 1060 px.
# Cada lista define el contorno de un sector del master plan.
SECTORES_SYNERGY_PIXELES = {
    "C1": [
        (350, 820), (450, 820), (550, 940), (635, 1000),
        (550, 1010), (390, 1025), (330, 950),
    ],
    "C2A": [
        (350, 410), (300, 475), (200, 600), (280, 660), (330, 555),
    ],
    "C2B": [
        (280, 560), (390, 580), (410, 710), (330, 760), (260, 650),
    ],
    "C2C": [
        (455, 397), (405, 395), (350, 430),
        (330, 550), (365, 565), (405, 400),
    ],
    "C3": [
        (390, 580), (520, 620), (490, 710), (410, 710),
    ],
    "C4": [
        (330, 760), (410, 710), (490, 710), (450, 900), (350, 920),
    ],
    "C5": [
        (520, 620), (600, 660), (570, 780), (490, 710),
    ],
    "C6A": [
        (405, 400), (610, 420), (575, 625), (390, 580),
    ],
    "C6B": [
        (475, 55), (615, 105), (603, 405), (455, 397), (450, 250),
    ],
    "C7": [
        (615, 105), (690, 130), (665, 380), (610, 420), (603, 200),
    ],
    "C8A": [
        (600, 660), (720, 650), (760, 850),
        (650, 1000), (550, 1000), (570, 780),
    ],
    "C8B": [
        (610, 420), (760, 390), (720, 650), (600, 660),
    ],
    "C9A": [
        (760, 390), (910, 400), (865, 660), (720, 650),
    ],
    "C9B": [
        (720, 650), (865, 660), (820, 850), (760, 850),
    ],
    "C10": [
        (410, 710), (490, 710), (570, 780), (530, 830), (450, 820),
    ],
    "Amenities": [
        (260, 650), (330, 760), (350, 820), (300, 850), (250, 750),
    ],
    "Solar Park": [
        (690, 130), (930, 210), (910, 400), (760, 390), (665, 380),
    ],
}

ORDEN_SECTORES_SYNERGY = [
    "C1", "C2A", "C2B", "C2C", "C3", "C4", "C5",
    "C6A", "C6B", "C7", "C8A", "C8B", "C9A", "C9B",
    "C10", "Amenities", "Solar Park",
]


def convertir_pixel_a_coordenada_synergy(x, y):
    """Convierte un punto de la captura de Google Earth a latitud/longitud."""
    t = TRANSFORMACION_SYNERGY
    longitud = t["lon_x"] * x + t["lon_y"] * y + t["lon_c"]
    latitud = t["lat_x"] * x + t["lat_y"] * y + t["lat_c"]
    return latitud, longitud


def preparar_geojson_synergy():
    """Construye el FeatureCollection y el lookup de sectores de Synergy."""
    features = []
    lookup = {}

    for nombre_sector in ORDEN_SECTORES_SYNERGY:
        puntos_pixel = SECTORES_SYNERGY_PIXELES[nombre_sector]
        anillo = []

        for x, y in puntos_pixel:
            latitud, longitud = convertir_pixel_a_coordenada_synergy(x, y)
            anillo.append([longitud, latitud])

        if anillo[0] != anillo[-1]:
            anillo.append(anillo[0])

        feature = {
            "type": "Feature",
            "properties": {
                "sector_nombre": nombre_sector,
                "fuente": "Digitalización aproximada de imagen Google Earth",
            },
            "geometry": {
                "type": "Polygon",
                "coordinates": [anillo],
            },
        }

        # Corrige pequeños cruces accidentales en los vértices aproximados.
        geometria = shape(feature["geometry"])
        if not geometria.is_valid:
            geometria = geometria.buffer(0)
            feature["geometry"] = mapping(geometria)

        features.append(feature)
        lookup[nombre_sector] = feature

    return {
        "type": "FeatureCollection",
        "features": features,
    }, lookup


geojson_synergy, synergy_lookup = preparar_geojson_synergy()
sectores_empresa = ORDEN_SECTORES_SYNERGY.copy()

# Geometría global para encuadrar las dos áreas en el mapa.
geometria_synergy_total = unary_union(
    [shape(feature["geometry"]) for feature in geojson_synergy["features"]]
)

# Mantiene un contorno global disponible para funciones auxiliares.
contorno_synergy = geometria_synergy_total.convex_hull
AREA_EMPRESA_POLIGONO = [
    [latitud, longitud]
    for longitud, latitud in contorno_synergy.exterior.coords
]

# Centros de los sectores; se usan para pequeños puntos de referencia.
SECTORES_EMPRESA = {}
for nombre_sector, feature in synergy_lookup.items():
    centro = shape(feature["geometry"]).representative_point()
    SECTORES_EMPRESA[nombre_sector] = {
        "lat": centro.y,
        "lon": centro.x,
    }


# 3. ESTADO DE LA APLICACIÓN
proyectos_reforestacion = []
capas_proyectos = []
poligono_actual_geojson = None
area_actual_m2 = 0.0
ignorando_evento_dibujo = False


# 4. CÁLCULO DEL ÁREA REAL DIBUJADA
RADIO_TIERRA_M = 6_371_008.8


def calcular_area_anillo_m2(coordenadas):
    """Calcula el área aproximada de un anillo GeoJSON en m².

    GeoJSON usa el orden [longitud, latitud]. Para polígonos pequeños se utiliza
    una proyección local equirectangular, suficiente para este prototipo.
    """
    if not coordenadas or len(coordenadas) < 3:
        return 0.0

    puntos = [list(punto[:2]) for punto in coordenadas]
    if puntos[0] != puntos[-1]:
        puntos.append(puntos[0])

    latitud_media = sum(punto[1] for punto in puntos[:-1]) / max(1, len(puntos) - 1)
    cos_latitud = math.cos(math.radians(latitud_media))

    puntos_metros = [
        (
            RADIO_TIERRA_M * math.radians(longitud) * cos_latitud,
            RADIO_TIERRA_M * math.radians(latitud),
        )
        for longitud, latitud in puntos
    ]

    suma = 0.0
    for indice in range(len(puntos_metros) - 1):
        x1, y1 = puntos_metros[indice]
        x2, y2 = puntos_metros[indice + 1]
        suma += x1 * y2 - x2 * y1

    return abs(suma) / 2.0


def calcular_area_geometria_m2(geometria):
    """Calcula el área de una geometría Polygon o MultiPolygon de GeoJSON."""
    if not geometria:
        return 0.0

    tipo = geometria.get("type")
    coordenadas = geometria.get("coordinates", [])

    if tipo == "Polygon":
        if not coordenadas:
            return 0.0

        area_exterior = calcular_area_anillo_m2(coordenadas[0])
        area_huecos = sum(
            calcular_area_anillo_m2(anillo)
            for anillo in coordenadas[1:]
        )
        return max(0.0, area_exterior - area_huecos)

    if tipo == "MultiPolygon":
        return sum(
            calcular_area_geometria_m2(
                {"type": "Polygon", "coordinates": poligono}
            )
            for poligono in coordenadas
        )

    return 0.0


def normalizar_feature_dibujo(valor):
    """Convierte la respuesta del DrawControl en una Feature GeoJSON."""
    if not valor:
        return None

    if valor.get("type") == "Feature":
        return valor

    if valor.get("type") == "FeatureCollection":
        features = valor.get("features", [])
        return features[-1] if features else None

    if valor.get("type") in {"Polygon", "MultiPolygon"}:
        return {
            "type": "Feature",
            "properties": {},
            "geometry": valor,
        }

    return None


def formatear_area_dibujada(area_m2):
    hectareas = area_m2 / 10_000
    manzanas = area_m2 / M2_POR_MANZANA

    return f"""
    <div style="
        background:#E8F5E9;
        border-left:5px solid #2E8B57;
        padding:12px 14px;
        border-radius:5px;
        font-family:Arial;
        line-height:1.55;
        max-width:700px;
    ">
        <b style="color:#0B5D3B;">Área real capturada correctamente</b><br>
        <b>{area_m2:,.2f} m²</b> · {hectareas:,.4f} ha · {manzanas:,.4f} mz
        <br>
        <span style="font-size:12px; color:#455A64;">
            Puedes editar el polígono con la herramienta del mapa antes de guardar.
        </span>
    </div>
    """



# 5. CONTROLES DEL FORMULARIO
# Cada proyecto puede contener varios tipos de árbol.
filas_arboles = []

contenedor_arboles = widgets.VBox(
    layout=widgets.Layout(
        width="760px",
        max_width="100%",
        gap="7px",
    )
)

mensaje_arboles = widgets.HTML(value="")

agregar_tipo_arbol_button = widgets.Button(
    description="Agregar otro tipo de árbol",
    button_style="",
    icon="plus",
    layout=widgets.Layout(width="245px", height="38px"),
)


def refrescar_contenedor_arboles():
    """Actualiza visualmente las filas de especies registradas."""
    contenedor_arboles.children = tuple(
        registro["fila"] for registro in filas_arboles
    )


def crear_fila_arbol(tipo=None, cantidad=1):
    """Crea una fila con especie, cantidad y botón para eliminarla."""
    tipos_disponibles = list(tree_impact_data.keys())
    tipo = tipo if tipo in tree_impact_data else tipos_disponibles[0]

    selector = widgets.Dropdown(
        options=tipos_disponibles,
        value=tipo,
        description="",
        layout=widgets.Layout(width="290px"),
    )

    cantidad_input = widgets.BoundedIntText(
        value=max(1, int(cantidad)),
        min=1,
        max=100000,
        step=1,
        description="",
        layout=widgets.Layout(width="165px"),
    )

    eliminar_button = widgets.Button(
        description="Eliminar",
        icon="trash",
        button_style="danger",
        layout=widgets.Layout(width="115px", height="36px"),
    )

    fila = widgets.HBox(
        [selector, cantidad_input, eliminar_button],
        layout=widgets.Layout(
            display="flex",
            flex_flow="row wrap",
            align_items="center",
            gap="8px",
        ),
    )

    registro = {
        "selector": selector,
        "cantidad_input": cantidad_input,
        "eliminar_button": eliminar_button,
        "fila": fila,
    }

    # Las funciones se resuelven al momento de ejecutar el evento.
    selector.observe(lambda change: actualizar_preview_impacto(change), names="value")
    cantidad_input.observe(lambda change: actualizar_preview_impacto(change), names="value")
    eliminar_button.on_click(
        lambda boton, registro_actual=registro: eliminar_fila_arbol(registro_actual)
    )

    filas_arboles.append(registro)
    refrescar_contenedor_arboles()
    return registro


def eliminar_fila_arbol(registro):
    """Elimina una especie, manteniendo al menos una fila activa."""
    if len(filas_arboles) <= 1:
        mensaje_arboles.value = """
        <div style="color:#B71C1C; font-family:Arial; margin-top:5px;">
            Debe quedar al menos un tipo de árbol en el proyecto.
        </div>
        """
        return

    if registro in filas_arboles:
        filas_arboles.remove(registro)

    mensaje_arboles.value = ""
    refrescar_contenedor_arboles()
    actualizar_preview_impacto()


def agregar_fila_arbol(b=None):
    """Agrega una nueva especie al formulario."""
    usados = {
        registro["selector"].value
        for registro in filas_arboles
    }
    tipo_sugerido = next(
        (
            tipo
            for tipo in tree_impact_data
            if tipo not in usados
        ),
        list(tree_impact_data.keys())[0],
    )

    crear_fila_arbol(tipo_sugerido, 1)
    mensaje_arboles.value = ""
    actualizar_preview_impacto()


def reiniciar_filas_arboles(cantidad_inicial=1):
    """Restablece el formulario a una sola especie."""
    filas_arboles.clear()
    crear_fila_arbol(
        list(tree_impact_data.keys())[0],
        cantidad_inicial,
    )
    mensaje_arboles.value = ""
    actualizar_preview_impacto()


def obtener_arboles_formulario():
    """Devuelve las especies del formulario y consolida duplicados."""
    acumulado = {}
    orden = []

    for registro in filas_arboles:
        tipo = registro["selector"].value
        cantidad = int(registro["cantidad_input"].value)

        if tipo not in acumulado:
            acumulado[tipo] = 0
            orden.append(tipo)

        acumulado[tipo] += cantidad

    arboles = []
    for tipo in orden:
        cantidad = acumulado[tipo]
        impacto_por_arbol = tree_impact_data[tipo]
        arboles.append(
            {
                "tipo": tipo,
                "cantidad": cantidad,
                "impacto_por_arbol": impacto_por_arbol,
                "impacto_estimado_m2": cantidad * impacto_por_arbol,
            }
        )

    return arboles


# Primera fila visible al cargar el formulario.
crear_fila_arbol("Pino", 100)


area_selector = widgets.Dropdown(
    options=["Ciudad de Guatemala", "Synergy"],
    value="Ciudad de Guatemala",
    description="Área general:",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="520px"),
)


location_selector = widgets.Dropdown(
    options=zone_names,
    value=zone_names[0],
    description="Zona:",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="520px"),
)


project_date_picker = widgets.DatePicker(
    description="Fecha del proyecto:",
    value=date.today(),
    max=date.today(),
    style={"description_width": "150px"},
    layout=widgets.Layout(width="520px"),
)


form_message = widgets.HTML(value="")
area_dibujada_html = widgets.HTML(
    value="""
    <div style="
        background:#FFF8E1;
        border-left:5px solid #F9A825;
        padding:12px 14px;
        border-radius:5px;
        font-family:Arial;
        max-width:700px;
    ">
        <b>Área pendiente:</b> usa la herramienta de polígono o rectángulo del mapa.
    </div>
    """
)

impacto_preview_html = widgets.HTML(value="")


add_button = widgets.Button(
    description="Agregar proyecto",
    button_style="success",
    icon="plus",
    layout=widgets.Layout(width="210px", height="42px"),
)

clear_drawing_button = widgets.Button(
    description="Borrar área dibujada",
    button_style="warning",
    icon="eraser",
    layout=widgets.Layout(width="220px", height="42px"),
)

reset_form_button = widgets.Button(
    description="Limpiar formulario",
    button_style="info",
    icon="refresh",
    layout=widgets.Layout(width="210px", height="42px"),
)

reset_all_button = widgets.Button(
    description="Reiniciar todo",
    button_style="danger",
    icon="trash",
    layout=widgets.Layout(width="210px", height="42px"),
)

view_all_button = widgets.Button(
    description="Ver ambas áreas",
    icon="globe",
    layout=widgets.Layout(width="180px", height="38px"),
)

center_selection_button = widgets.Button(
    description="Centrar selección",
    icon="crosshairs",
    layout=widgets.Layout(width="180px", height="38px"),
)


dashboard_output = widgets.Output(
    layout=widgets.Layout(width="100%", overflow="visible")
)


# 6. MAPA INTERACTIVO Y CAPAS DE REFERENCIA
mapa = Map(
    center=(14.49, -90.63),
    zoom=10,
    basemap=basemaps.CartoDB.Positron,
    scroll_wheel_zoom=True,
    layout=widgets.Layout(width="100%", height="620px"),
)

mapa.add_control(FullScreenControl(position="topright"))
mapa.add_control(ScaleControl(position="bottomleft", metric=True, imperial=False))
mapa.add_control(LayersControl(position="topright"))


capa_zonas_referencia = GeoJSON(
    data=geojson_zonas,
    name="Zonas de Ciudad de Guatemala",
    style={
        "fillColor": "#F2F2F2",
        "color": "#607D8B",
        "weight": 1.2,
        "fillOpacity": 0.08,
    },
    hover_style={
        "fillColor": "#DFF3E7",
        "color": "#087F23",
        "weight": 2.5,
        "fillOpacity": 0.25,
    },
)
mapa.add_layer(capa_zonas_referencia)


capa_sectores_synergy = GeoJSON(
    data=geojson_synergy,
    name="Sectores del master plan de Synergy",
    style={
        "fillColor": "#EAF5EC",
        "color": "#0B5D3B",
        "weight": 2.2,
        "fillOpacity": 0.16,
    },
    hover_style={
        "fillColor": "#9CCC65",
        "color": "#173B57",
        "weight": 3.2,
        "fillOpacity": 0.38,
    },
)
mapa.add_layer(capa_sectores_synergy)


# Puntos centrales discretos para facilitar la lectura de cada sector.
capas_referencia_sector = []
for nombre_sector, coordenadas in SECTORES_EMPRESA.items():
    marcador_sector = CircleMarker(
        location=(coordenadas["lat"], coordenadas["lon"]),
        radius=4,
        color="#173B57",
        weight=2,
        fill_color="#FFFFFF",
        fill_opacity=0.95,
    )
    marcador_sector.popup = widgets.HTML(
        value=f"<b style='font-family:Arial;'>Sector {nombre_sector}</b>"
    )
    capas_referencia_sector.append(marcador_sector)


grupo_sectores = LayerGroup(
    layers=tuple(capas_referencia_sector),
    name="Centros de sectores Synergy",
)
mapa.add_layer(grupo_sectores)


grupo_seleccion = LayerGroup(name="Ubicación seleccionada")
grupo_proyectos = LayerGroup(name="Áreas reforestadas registradas")
mapa.add_layer(grupo_seleccion)
mapa.add_layer(grupo_proyectos)


draw_control = DrawControl(
    position="topleft",
    polygon={
        "shapeOptions": {
            "color": "#0B5D3B",
            "weight": 3,
            "fillColor": "#2E8B57",
            "fillOpacity": 0.45,
        }
    },
    rectangle={
        "shapeOptions": {
            "color": "#173B57",
            "weight": 3,
            "fillColor": "#4CAF50",
            "fillOpacity": 0.40,
        }
    },
    polyline={},
    circle={},
    circlemarker={},
    marker={},
    edit=True,
    remove=True,
)
mapa.add_control(draw_control)


# 7. FUNCIONES DEL MAPA Y DEL DIBUJO
def ver_ambas_areas(b=None):
    geometria_ciudad = unary_union(
        [shape(feature["geometry"]) for feature in geojson_zonas["features"]]
    )
    min_lon_ciudad, min_lat_ciudad, max_lon_ciudad, max_lat_ciudad = geometria_ciudad.bounds

    min_lon_synergy, min_lat_synergy, max_lon_synergy, max_lat_synergy = (
        geometria_synergy_total.bounds
    )

    min_lat = min(min_lat_ciudad, min_lat_synergy)
    max_lat = max(max_lat_ciudad, max_lat_synergy)
    min_lon = min(min_lon_ciudad, min_lon_synergy)
    max_lon = max(max_lon_ciudad, max_lon_synergy)

    mapa.fit_bounds([[min_lat, min_lon], [max_lat, max_lon]])


def limpiar_dibujo_actual(b=None, limpiar_control=True):
    global poligono_actual_geojson, area_actual_m2, ignorando_evento_dibujo

    poligono_actual_geojson = None
    area_actual_m2 = 0.0

    area_dibujada_html.value = """
    <div style="
        background:#FFF8E1;
        border-left:5px solid #F9A825;
        padding:12px 14px;
        border-radius:5px;
        font-family:Arial;
        max-width:700px;
    ">
        <b>Área pendiente:</b> usa la herramienta de polígono o rectángulo del mapa.
    </div>
    """

    if limpiar_control:
        ignorando_evento_dibujo = True
        try:
            if hasattr(draw_control, "clear"):
                draw_control.clear()
            else:
                draw_control.data = []
        except Exception:
            try:
                draw_control.data = []
            except Exception:
                pass
        finally:
            ignorando_evento_dibujo = False

    actualizar_preview_impacto()


def obtener_ultimo_dibujo(evento_geojson=None):
    datos = list(getattr(draw_control, "data", []) or [])
    if datos:
        return normalizar_feature_dibujo(datos[-1])
    return normalizar_feature_dibujo(evento_geojson or {})


def manejar_dibujo(target, action, geo_json):
    global poligono_actual_geojson, area_actual_m2, ignorando_evento_dibujo

    if ignorando_evento_dibujo:
        return

    if action == "deleted" and not getattr(draw_control, "data", []):
        limpiar_dibujo_actual(limpiar_control=False)
        return

    feature = obtener_ultimo_dibujo(geo_json)
    if feature is None:
        limpiar_dibujo_actual(limpiar_control=False)
        return

    geometria = feature.get("geometry", {})
    if geometria.get("type") not in {"Polygon", "MultiPolygon"}:
        form_message.value = """
        <div style="color:#B71C1C; margin:8px 0; font-family:Arial;">
            <b>Debes dibujar un polígono o rectángulo.</b>
        </div>
        """
        return

    area_calculada = calcular_area_geometria_m2(geometria)
    if area_calculada <= 0:
        form_message.value = """
        <div style="color:#B71C1C; margin:8px 0; font-family:Arial;">
            <b>No fue posible calcular el área. Vuelve a dibujar el polígono.</b>
        </div>
        """
        return

    poligono_actual_geojson = copy.deepcopy(feature)
    poligono_actual_geojson.setdefault("properties", {})
    area_actual_m2 = area_calculada
    area_dibujada_html.value = formatear_area_dibujada(area_actual_m2)
    form_message.value = ""

    # Se conserva solamente el último dibujo para evitar guardar dos áreas
    # accidentales dentro del mismo proyecto.
    try:
        if len(draw_control.data) > 1:
            ignorando_evento_dibujo = True
            draw_control.data = [copy.deepcopy(feature)]
            ignorando_evento_dibujo = False
    except Exception:
        ignorando_evento_dibujo = False

    actualizar_preview_impacto()


def obtener_zona(nombre_zona):
    return zone_lookup.get(nombre_zona)


def actualizar_opciones_ubicacion(change=None):
    if area_selector.value == "Ciudad de Guatemala":
        location_selector.options = zone_names
        location_selector.description = "Zona:"
        if zone_names:
            location_selector.value = zone_names[0]
    else:
        location_selector.options = sectores_empresa
        location_selector.description = "Sector:"
        if sectores_empresa:
            location_selector.value = sectores_empresa[0]

    limpiar_dibujo_actual()
    actualizar_seleccion_mapa()


def actualizar_seleccion_mapa(change=None, centrar=True):
    capas = []
    area_general = area_selector.value
    ubicacion = location_selector.value

    if area_general == "Ciudad de Guatemala":
        feature_zona = obtener_zona(ubicacion)
        if feature_zona is not None:
            capa = GeoJSON(
                data=feature_zona,
                style={
                    "fillColor": "#DFF3E7",
                    "color": "#087F23",
                    "weight": 3,
                    "fillOpacity": 0.28,
                },
            )
            capas.append(capa)

            if centrar:
                min_lon, min_lat, max_lon, max_lat = shape(feature_zona["geometry"]).bounds
                mapa.fit_bounds([[min_lat, min_lon], [max_lat, max_lon]])

    else:
        feature_sector = synergy_lookup.get(ubicacion)
        if feature_sector is not None:
            capa = GeoJSON(
                data=feature_sector,
                style={
                    "fillColor": "#9CCC65",
                    "color": "#173B57",
                    "weight": 4,
                    "fillOpacity": 0.42,
                },
            )
            capas.append(capa)

            if centrar:
                min_lon, min_lat, max_lon, max_lat = shape(
                    feature_sector["geometry"]
                ).bounds
                mapa.fit_bounds([[min_lat, min_lon], [max_lat, max_lon]])

    grupo_seleccion.layers = tuple(capas)


def centrar_seleccion(b=None):
    actualizar_seleccion_mapa(centrar=True)


def actualizar_preview_impacto(change=None):
    arboles = obtener_arboles_formulario()
    total_arboles = sum(arbol["cantidad"] for arbol in arboles)
    impacto_estimado = sum(
        arbol["impacto_estimado_m2"]
        for arbol in arboles
    )

    texto_area = (
        f"{area_actual_m2:,.2f} m²"
        if area_actual_m2 > 0
        else "pendiente de dibujar"
    )

    filas_detalle = "".join(
        f"""
        <tr>
            <td style="padding:5px 8px; border-bottom:1px solid #D9E1E5;">
                {arbol['tipo']}
            </td>
            <td style="padding:5px 8px; border-bottom:1px solid #D9E1E5; text-align:right;">
                {arbol['cantidad']:,}
            </td>
            <td style="padding:5px 8px; border-bottom:1px solid #D9E1E5; text-align:right;">
                {arbol['impacto_estimado_m2']:,.0f} m²
            </td>
        </tr>
        """
        for arbol in arboles
    )

    impacto_preview_html.value = f"""
    <div style="
        background:#EEF4F7;
        border-left:5px solid #173B57;
        padding:10px 14px;
        border-radius:5px;
        font-family:Arial;
        max-width:760px;
        line-height:1.5;
    ">
        <b>Vista previa del proyecto:</b><br>
        Área física real: <b>{texto_area}</b><br>
        Total de árboles: <b>{total_arboles:,}</b><br>
        Cobertura/impacto arbóreo estimado total:
        <b>{impacto_estimado:,.0f} m²</b>

        <table style="
            width:100%;
            border-collapse:collapse;
            margin-top:8px;
            font-size:12px;
        ">
            <tr style="background:#DCE7ED;">
                <th style="padding:5px 8px; text-align:left;">Tipo</th>
                <th style="padding:5px 8px; text-align:right;">Cantidad</th>
                <th style="padding:5px 8px; text-align:right;">Impacto estimado</th>
            </tr>
            {filas_detalle}
        </table>
    </div>
    """


draw_control.on_draw(manejar_dibujo)


# 8. REGISTRO Y VISUALIZACIÓN DE PROYECTOS
def obtener_color_fase(numero_fase):
    return FASES_REFORESTACION[numero_fase]["color_hex"]


def obtener_marker_color_fase(numero_fase):
    return FASES_REFORESTACION[numero_fase]["marker_color"]


def agregar_capas_proyecto_al_mapa(proyecto):
    feature = copy.deepcopy(proyecto["poligono_geojson"])
    fase = calcular_fase_reforestacion(proyecto["fecha_proyecto"])
    color = fase["color_hex"]

    feature.setdefault("properties", {})
    feature["properties"].update(
        {
            "proyecto": proyecto["numero"],
            "area_general": proyecto["area_general"],
            "ubicacion": proyecto["ubicacion"],
            "area_real_m2": proyecto["area_real_m2"],
        }
    )

    capa_poligono = GeoJSON(
        data=feature,
        style={
            "fillColor": color,
            "color": color,
            "weight": 3,
            "fillOpacity": 0.48,
        },
        hover_style={
            "fillColor": color,
            "color": "#173B57",
            "weight": 4,
            "fillOpacity": 0.68,
        },
    )

    geometria = shape(feature["geometry"])
    centro = geometria.representative_point()

    detalle_arboles_popup = "".join(
        f"""
        <li>
            <b>{arbol['tipo']}:</b> {arbol['cantidad']:,} árboles
            ({arbol['impacto_estimado_m2']:,.0f} m² estimados)
        </li>
        """
        for arbol in proyecto["arboles"]
    )

    popup_html = widgets.HTML(
        value=f"""
        <div style="font-family:Arial; width:340px; line-height:1.5;">
            <h4 style="color:{color}; margin:0 0 8px 0;">
                Proyecto {proyecto['numero']}
            </h4>
            <b>Área general:</b> {proyecto['area_general']}<br>
            <b>{proyecto['tipo_ubicacion']}:</b> {proyecto['ubicacion']}<br>
            <b>Fecha:</b> {formatear_fecha(proyecto['fecha_proyecto'])}<br>
            <b>Fase:</b> {fase['nombre']}<br>

            <b>Composición de árboles:</b>
            <ul style="margin:4px 0 8px 18px; padding:0;">
                {detalle_arboles_popup}
            </ul>
            <b>Total de árboles:</b> {proyecto['cantidad_total']:,}<br>

            <hr>
            <b>Área real dibujada:</b> {proyecto['area_real_m2']:,.2f} m²<br>
            <b>Hectáreas:</b> {proyecto['area_real_hectareas']:,.4f} ha<br>
            <b>Manzanas:</b> {proyecto['area_real_manzanas']:,.4f} mz<br>
            <b>Impacto arbóreo estimado:</b> {proyecto['impacto_estimado_m2']:,.0f} m²
        </div>
        """
    )

    icono = AwesomeIcon(
        name="tree",
        marker_color=fase["marker_color"],
        icon_color="white",
    )

    marcador = Marker(
        location=(centro.y, centro.x),
        icon=icono,
        title=f"Proyecto {proyecto['numero']} - {proyecto['ubicacion']}",
        draggable=False,
    )
    marcador.popup = popup_html

    capas_proyectos.extend([capa_poligono, marcador])
    grupo_proyectos.layers = tuple(capas_proyectos)


def validar_poligono_seleccionado():
    """Devuelve una advertencia si el centro del dibujo queda fuera de la referencia."""
    if poligono_actual_geojson is None:
        return ""

    try:
        geometria_proyecto = shape(poligono_actual_geojson["geometry"])
        punto_central = geometria_proyecto.representative_point()

        if area_selector.value == "Ciudad de Guatemala":
            feature_zona = obtener_zona(location_selector.value)
            if feature_zona is None:
                return ""
            referencia = shape(feature_zona["geometry"])
        else:
            feature_sector = synergy_lookup.get(location_selector.value)
            if feature_sector is None:
                return ""
            referencia = shape(feature_sector["geometry"])

        if not referencia.buffer(0.002).contains(punto_central):
            return (
                "El centro del polígono está fuera de la ubicación de referencia. "
                "Puedes guardarlo, pero conviene revisar la selección."
            )
    except Exception:
        return ""

    return ""


def agregar_proyecto(b=None):
    global poligono_actual_geojson, area_actual_m2

    fecha_proyecto = project_date_picker.value

    if fecha_proyecto is None:
        form_message.value = """
        <div style="color:#B71C1C; margin:8px 0; font-family:Arial;">
            <b>Selecciona la fecha en que se realizó el proyecto.</b>
        </div>
        """
        return

    if fecha_proyecto > date.today():
        form_message.value = """
        <div style="color:#B71C1C; margin:8px 0; font-family:Arial;">
            <b>La fecha del proyecto no puede estar en el futuro.</b>
        </div>
        """
        return

    if poligono_actual_geojson is None or area_actual_m2 <= 0:
        form_message.value = """
        <div style="color:#B71C1C; margin:8px 0; font-family:Arial;">
            <b>Dibuja primero el área realmente reforestada en el mapa.</b>
        </div>
        """
        return

    try:
        geometria_shapely = shape(poligono_actual_geojson["geometry"])
        if geometria_shapely.is_empty:
            raise ValueError("Geometría vacía")
        if not geometria_shapely.is_valid:
            geometria_shapely = geometria_shapely.buffer(0)
        if geometria_shapely.is_empty:
            raise ValueError("Geometría inválida")
    except Exception:
        form_message.value = """
        <div style="color:#B71C1C; margin:8px 0; font-family:Arial;">
            <b>El polígono no es válido. Bórralo y vuelve a dibujarlo.</b>
        </div>
        """
        return

    arboles = obtener_arboles_formulario()
    if not arboles:
        form_message.value = """
        <div style="color:#B71C1C; margin:8px 0; font-family:Arial;">
            <b>Agrega al menos un tipo de árbol.</b>
        </div>
        """
        return

    cantidad_total = sum(arbol["cantidad"] for arbol in arboles)
    impacto_estimado_m2 = sum(
        arbol["impacto_estimado_m2"]
        for arbol in arboles
    )

    area_general = area_selector.value
    ubicacion = location_selector.value
    tipo_ubicacion = "Zona" if area_general == "Ciudad de Guatemala" else "Sector"

    advertencia = validar_poligono_seleccionado()

    nuevo_proyecto = {
        "numero": len(proyectos_reforestacion) + 1,
        "area_general": area_general,
        "ubicacion": ubicacion,
        "tipo_ubicacion": tipo_ubicacion,
        "arboles": copy.deepcopy(arboles),
        "cantidad_total": cantidad_total,
        "fecha_proyecto": fecha_proyecto.isoformat(),
        "impacto_estimado_m2": impacto_estimado_m2,
        "poligono_geojson": copy.deepcopy(poligono_actual_geojson),
        "area_real_m2": area_actual_m2,
        "area_real_hectareas": area_actual_m2 / 10_000,
        "area_real_manzanas": area_actual_m2 / M2_POR_MANZANA,
    }

    proyectos_reforestacion.append(nuevo_proyecto)
    agregar_capas_proyecto_al_mapa(nuevo_proyecto)
    actualizar_dashboard()

    if advertencia:
        form_message.value = f"""
        <div style="
            color:#7A4F00;
            background:#FFF8E1;
            border-left:5px solid #F9A825;
            padding:10px 12px;
            margin:8px 0;
            font-family:Arial;
        ">
            <b>Proyecto guardado.</b> {advertencia}
        </div>
        """
    else:
        form_message.value = """
        <div style="
            color:#0B5D3B;
            background:#E8F5E9;
            border-left:5px solid #2E8B57;
            padding:10px 12px;
            margin:8px 0;
            font-family:Arial;
        ">
            <b>Proyecto guardado correctamente.</b>
        </div>
        """

    limpiar_dibujo_actual()
    reiniciar_filas_arboles(cantidad_inicial=1)
    project_date_picker.value = date.today()


def actualizar_dashboard():
    with dashboard_output:
        clear_output(wait=True)

        if not proyectos_reforestacion:
            display(
                widgets.HTML(
                    value="""
                    <div style="
                        background:#F5F5F5;
                        padding:15px;
                        border-left:5px solid #43A047;
                        max-width:950px;
                        font-family:Arial;
                    ">
                        Todavía no se han agregado proyectos.
                    </div>
                    """
                )
            )
            return

        total_proyectos = len(proyectos_reforestacion)
        total_arboles = sum(proyecto["cantidad_total"] for proyecto in proyectos_reforestacion)
        total_area_real_m2 = sum(proyecto["area_real_m2"] for proyecto in proyectos_reforestacion)
        total_impacto_estimado_m2 = sum(
            proyecto["impacto_estimado_m2"] for proyecto in proyectos_reforestacion
        )
        total_hectareas = total_area_real_m2 / 10_000
        total_manzanas = total_area_real_m2 / M2_POR_MANZANA

        ubicaciones_intervenidas = {
            (proyecto["area_general"], proyecto["ubicacion"])
            for proyecto in proyectos_reforestacion
        }

        conteo_fases = {1: 0, 2: 0, 3: 0}
        filas_tabla = ""

        for proyecto in proyectos_reforestacion:
            fase = calcular_fase_reforestacion(proyecto["fecha_proyecto"])
            conteo_fases[fase["numero"]] += 1

            composicion_arboles = "<br>".join(
                f"{arbol['tipo']}: {arbol['cantidad']:,}"
                for arbol in proyecto["arboles"]
            )

            filas_tabla += f"""
            <tr>
                <td style="padding:8px; border:1px solid #DDD;">{proyecto['numero']}</td>
                <td style="padding:8px; border:1px solid #DDD;">{proyecto['area_general']}</td>
                <td style="padding:8px; border:1px solid #DDD;">{proyecto['ubicacion']}</td>
                <td style="padding:8px; border:1px solid #DDD; white-space:nowrap;">{formatear_fecha(proyecto['fecha_proyecto'])}</td>
                <td style="padding:8px; border:1px solid #DDD;">
                    <span style="
                        display:inline-block;
                        background:{fase['color_hex']};
                        color:white;
                        padding:4px 8px;
                        border-radius:12px;
                        white-space:nowrap;
                    ">
                        {fase['nombre']}
                    </span>
                </td>
                <td style="padding:8px; border:1px solid #DDD; line-height:1.45;">{composicion_arboles}</td>
                <td style="padding:8px; border:1px solid #DDD; text-align:right;">{proyecto['cantidad_total']:,}</td>
                <td style="padding:8px; border:1px solid #DDD; text-align:right;">{proyecto['area_real_m2']:,.2f}</td>
                <td style="padding:8px; border:1px solid #DDD; text-align:right;">{proyecto['impacto_estimado_m2']:,.0f}</td>
            </tr>
            """

        panel_comparacion = crear_panel_comparacion_superficies(total_area_real_m2)

        display(
            widgets.HTML(
                value=f"""
                <div style="
                    background:#E8F5E9;
                    border-left:6px solid #087F23;
                    padding:16px;
                    margin:18px 0 16px 0;
                    font-family:Arial;
                    max-width:950px;
                    border-radius:4px;
                ">
                    <h3 style="color:#087F23; margin-top:0; margin-bottom:10px;">
                        Resumen general de reforestación
                    </h3>

                    <b>Proyectos registrados:</b> {total_proyectos}<br>
                    <b>Ubicaciones intervenidas:</b> {len(ubicaciones_intervenidas)}<br>
                    <b>Total de árboles:</b> {total_arboles:,}<br>
                    <b>Área física real:</b> {total_area_real_m2:,.2f} m²<br>
                    <b>Equivalente:</b> {total_hectareas:,.4f} ha · {total_manzanas:,.4f} mz<br>
                    <b>Cobertura/impacto arbóreo estimado:</b> {total_impacto_estimado_m2:,.0f} m²

                    <div style="display:flex; flex-wrap:wrap; gap:10px; margin-top:14px;">
                        <div style="background:#8BC34A; color:white; padding:10px 14px; border-radius:6px;">
                            <b>Establecimiento:</b> {conteo_fases[1]}
                        </div>
                        <div style="background:#2E8B57; color:white; padding:10px 14px; border-radius:6px;">
                            <b>Desarrollo:</b> {conteo_fases[2]}
                        </div>
                        <div style="background:#0B5D3B; color:white; padding:10px 14px; border-radius:6px;">
                            <b>Consolidación:</b> {conteo_fases[3]}
                        </div>
                    </div>

                    <div style="font-size:12px; margin-top:10px; color:#455A64;">
                        El área física proviene de los polígonos dibujados por el usuario.
                        El impacto arbóreo es una estimación independiente basada en la cantidad y los tipos de árbol.
                    </div>
                </div>

                {panel_comparacion}

                <div style="overflow-x:auto; max-width:1100px; margin-bottom:18px;">
                    <table style="border-collapse:collapse; width:100%; font-family:Arial; font-size:13px; min-width:1050px;">
                        <tr style="background:#087F23; color:white;">
                            <th style="padding:9px;">Proyecto</th>
                            <th style="padding:9px;">Área general</th>
                            <th style="padding:9px;">Zona / sector</th>
                            <th style="padding:9px;">Fecha</th>
                            <th style="padding:9px;">Fase</th>
                            <th style="padding:9px;">Composición de árboles</th>
                            <th style="padding:9px;">Total árboles</th>
                            <th style="padding:9px;">Área real m²</th>
                            <th style="padding:9px;">Impacto estimado m²</th>
                        </tr>
                        {filas_tabla}
                    </table>
                </div>
                """
            )
        )


def limpiar_formulario(b=None):
    reiniciar_filas_arboles(cantidad_inicial=1)
    area_selector.value = "Ciudad de Guatemala"
    actualizar_opciones_ubicacion()
    project_date_picker.value = date.today()
    form_message.value = ""
    limpiar_dibujo_actual()


def reiniciar_todo(b=None):
    proyectos_reforestacion.clear()
    capas_proyectos.clear()
    grupo_proyectos.layers = tuple()
    limpiar_formulario()
    actualizar_dashboard()
    ver_ambas_areas()


# 9. CONECTAR EVENTOS
area_selector.observe(actualizar_opciones_ubicacion, names="value")
location_selector.observe(actualizar_seleccion_mapa, names="value")
agregar_tipo_arbol_button.on_click(agregar_fila_arbol)

add_button.on_click(agregar_proyecto)
clear_drawing_button.on_click(limpiar_dibujo_actual)
reset_form_button.on_click(limpiar_formulario)
reset_all_button.on_click(reiniciar_todo)
view_all_button.on_click(ver_ambas_areas)
center_selection_button.on_click(centrar_seleccion)


# 10. INTERFAZ
titulo = widgets.HTML(
    value="""
    <div style="
        background:linear-gradient(90deg, #087F23, #43A047);
        color:white;
        padding:18px;
        border-radius:10px;
        max-width:1100px;
        font-family:Arial;
        margin-bottom:15px;
    ">
        <h2 style="margin:0 0 6px 0;">
            Spectrum | Seguimiento de áreas realmente reforestadas
        </h2>
        <div>
            Ciudad de Guatemala y área Synergy
        </div>
    </div>
    """
)


instrucciones = widgets.HTML(
    value="""
    <div style="
        background:#F5F5F5;
        padding:14px 16px;
        border-radius:6px;
        max-width:1070px;
        margin-bottom:15px;
        font-family:Arial;
        line-height:1.55;
    ">
        <b>Cómo registrar un proyecto:</b><br>
        1. Selecciona Ciudad de Guatemala o Synergy.<br>
        2. Agrega uno o varios tipos de árbol e indica la cantidad de cada uno.<br>
        3. Selecciona la zona o sector de referencia.<br>
        4. Usa el botón de <b>polígono</b> o <b>rectángulo</b> ubicado en la parte izquierda del mapa.<br>
        5. Marca el perímetro del terreno realmente reforestado y cierra la figura.<br>
        6. Revisa el área calculada y presiona <b>Agregar proyecto</b>.
        <br><br>
        La zona o sector sirve únicamente como referencia. El área verde guardada será exactamente
        el polígono dibujado por el usuario, no toda la zona.
    </div>
    """
)


formulario = widgets.VBox(
    [
        widgets.HTML(
            value="""
            <h3 style="color:#087F23; margin:0 0 10px 0; font-family:Arial;">
                Registrar nuevo proyecto
            </h3>
            """
        ),
        widgets.HTML(
            value="""
            <div style="font-family:Arial; margin:2px 0 5px 0;">
                <b>Composición de árboles</b><br>
                <span style="font-size:12px; color:#546E7A;">
                    Agrega una fila por especie. Si repites una especie, sus cantidades se consolidan.
                </span>
            </div>
            """
        ),
        widgets.HBox(
            [
                widgets.HTML(
                    value='<div style="width:290px; font-family:Arial; font-size:12px;"><b>Tipo de árbol</b></div>'
                ),
                widgets.HTML(
                    value='<div style="width:165px; font-family:Arial; font-size:12px;"><b>Cantidad</b></div>'
                ),
            ],
            layout=widgets.Layout(gap="8px"),
        ),
        contenedor_arboles,
        widgets.HBox(
            [agregar_tipo_arbol_button],
            layout=widgets.Layout(margin="3px 0 5px 0"),
        ),
        mensaje_arboles,
        area_selector,
        location_selector,
        project_date_picker,
        impacto_preview_html,
        area_dibujada_html,
        form_message,
        widgets.HBox(
            [center_selection_button, view_all_button, clear_drawing_button],
            layout=widgets.Layout(flex_flow="row wrap", gap="10px"),
        ),
        widgets.HBox(
            [add_button, reset_form_button, reset_all_button],
            layout=widgets.Layout(flex_flow="row wrap", gap="10px"),
        ),
    ],
    layout=widgets.Layout(
        border="2px solid #43A047",
        padding="16px",
        margin="0 0 16px 0",
        width="1100px",
        max_width="96%",
    ),
)


mapa_contenedor = widgets.VBox(
    [
        widgets.HTML(
            value="""
            <div style="
                background:#EEF4F7;
                border-left:5px solid #173B57;
                padding:10px 14px;
                font-family:Arial;
                max-width:1070px;
                margin-bottom:8px;
            ">
                <b>Dibuja aquí el perímetro real.</b> Puedes editar sus vértices o eliminarlo antes de guardar.
            </div>
            """
        ),
        mapa,
    ],
    layout=widgets.Layout(width="1100px", max_width="96%"),
)


actualizar_preview_impacto()
actualizar_seleccion_mapa(centrar=False)
actualizar_dashboard()
ver_ambas_areas()


display(titulo)
display(instrucciones)
display(formulario)
display(mapa_contenedor)
display(dashboard_output)


Cargando mapa de zonas de la Ciudad de Guatemala...
Advertencia: la fuente geográfica no incluyó estas zonas: [18]
Mapa cargado correctamente: 21 zonas disponibles.


HTML(value='\n    <div style="\n        background:linear-gradient(90deg, #087F23, #43A047);\n        color:wh…

HTML(value='\n    <div style="\n        background:#F5F5F5;\n        padding:14px 16px;\n        border-radius…

Output(layout=Layout(overflow='visible', width='100%'))